<a href="https://colab.research.google.com/github/kangG718/electricity_tariff_policy_simulation/blob/main/%EC%A0%84%EA%B8%B0%EC%9A%94%EA%B8%88%EC%A0%95%EC%B1%85_%EC%8B%9C%EB%AE%AC%EB%A0%88%EC%9D%B4%EC%85%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# <mark>⚡️ 산업별 전력 사용 데이터기반 전기요금 결정 프레임워크 </mark>
>전력 산업은 공공 인프라 성격이 강해 요금이 완전히 시장에 의해 결정되기보다는 정부 정책과 규제를 통해 관리되는 경우가 많다. 전기요금이 지나치게 낮을 경우 발전·송배전 설비 투자와 안정적인 전력 공급이 어려워질 수 있기 때문에 정부는 생산자의 수익성과 소비자 부담 사이의 균형을 고려하여 요금 정책을 조정한다. 이러한 맥락에서 정부가 생산자의 매출을 증가시키기 위한 정책을 가정해 볼 수 있으며, 본 분석에서는 전기요금 지출이 매우 큰 초대형 소비 집단(VVIP)이 나타나는 시점을 중심으로 해당 집단의 전력 사용 특성을 분석한다. 이를 위해 VVIP가 존재하는 년월을 식별하고, 각 시점에서의 가구수 평균, 1가구당 평균 사용량, 1가구당 평균 전기요금 등을 비교하여 전력 소비 구조를 파악한다. 이후 이러한 분석 결과를 바탕으로 특정 시점의 전력 수요 구조를 반영한 요금 정책 시나리오를 구성하고, 평균판매단가 조정에 따른 전기요금 변화와 매출 변화를 데이터 기반 시뮬레이션을 통해 분석한다. 이를 통해 초대형 전력 소비 산업(VVIP)의 기준을 설정하고, 산업 집단별 차등 요금 인상 정책이 전기요금 구조와 매출에 미치는 영향을 분석할 수 있는 데이터 기반 정책 결정 프레임워크를 제시한다.




---
## 0) 데이터셋 개요

* **데이터 형태**: 산업 단위의 **월별 전력 사용 집계 데이터**  
* **데이터 크기**: 총 **4473개 관측치 × 5개 변수**  
* **행(Row)**: 하나의 행은 특정 년월(YYYYMM)에 특정 **산업 집단의 전력 사용 정보를 집계한 기록**을 의미함

* **열(Column) 구성**

  * **년월(YYYYMM)**: 전력 사용이 기록된 기준 월  
  * **가구수**: 해당 산업 집단에서 전력을 사용한 **시설 또는 사업장의 수**를 의미함( 일반적인 가정의 수 개념이 아님)
  * **사용량**: 해당 월 동안 해당 산업 집단이 **총 사용한 전력량**  
  * **전기요금**: 해당 전력 사용에 대해 **지불된 총 전기요금**  
  * **평균판매단가**: 전력 1단위당 요금 수준을 나타내는 값으로 **전기요금 ÷ 사용량**으로 계산됨


In [1]:
import numpy as np

try:
    # Colab 환경일 경우 (업로드 사용)
    from google.colab import files
    import contextlib, io
    with contextlib.redirect_stdout(io.StringIO()):
        uploaded = files.upload()
    raw = np.loadtxt(list(uploaded.keys())[0], delimiter='\t')

except:
    # 로컬 / GitHub 환경일 경우 (파일 경로에서 직접 로드)
    raw = np.loadtxt('Industry_MonthlyElectricUsage.txt', delimiter='\t')

raw

array([[2.02001000e+05, 1.30000000e+01, 1.11140000e+04, 1.70977500e+06,
        1.54000000e+02],
       [2.02001000e+05, 5.05200000e+03, 1.66106290e+07, 2.29918341e+09,
        1.38000000e+02],
       [2.02001000e+05, 5.57000000e+03, 6.51106010e+07, 8.23555475e+09,
        1.27000000e+02],
       ...,
       [2.02012000e+05, 9.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00],
       [2.02012000e+05, 4.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00],
       [2.02012000e+05, 1.30000000e+01, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00]])




---




## 1) 데이터의 크기 확인하여 행과 열의 갯수 출력




In [2]:
raw.shape, raw.dtype

((4473, 5), dtype('float64'))



---


## 2) 데이터의 결측값 현황 확인
수집한 데이터의 결측값이 있는지, 어디에 있는지를 확인해야 해당 결측값을 어떻게 처리할 지 결정할 수 있음





In [3]:
mask = np.isnan(raw)
mask

array([[False, False, False, False, False],
       [False, False, False, False, False],
       [False, False, False, False, False],
       ...,
       [False, False, False, False, False],
       [False, False, False, False, False],
       [False, False, False, False, False]])

In [4]:
mask.sum()

np.int64(0)

만약 `mask.sum()`의 결과가 0이 아니었다면, 이는 데이터 내에 결측값이 존재한다는 의미이므로 먼저 결측값이 정확히 어느 행과 열에 위치하는지 확인해야 함.

예를 들어 `rows, cols = np.where(mask)`로 받은 뒤 `list(zip(rows, cols))`처럼 묶어 보면 (row, column) 좌표 형태로 결측 위치를 확인할 수 있으며 특정 값까지 확인하려면 `raw[rows, cols]`로 실제 데이터도 함께 검증할 수 있음 (정수배열 인덱싱)


이후 해당 결측값이 왜 발생했는지와 분석에 미치는 영향을 검토한 뒤, 단순 입력 누락인지 실제로 의미 있는 값의 부재인지를 구분하여 처리 방식을 결정함
  - **결측값이 별로 없음**: 해당 행을 제거 (열을 제거하지는 않음! 샘플 하나를 없애는 게 특성 하나 없애는 거 보다 영향력이 적으니까)
  - **중요한 변수이거나 결측 비중이 큼** : 평균, 중앙값, 혹은 주변 값으로 대체




---


## 3) 데이터의 사용량이 음수로 표시된 부분 확인하고 제거할 필요가 있음
> 데이터 값이 음수라는 사실 자체가 문제인 것은 아니지만, **사용량이 음수로 나타나는 경우는 도메인 지식에 비추어 볼 때 물리적으로 발생할 수 없는 값이므로 오류 데이터로 판단함.** 이처럼 데이터 오류의 기준은 절대적으로 고정된 것이 아니라, 분석 대상 데이터의 의미와 도메인 특성에 따라 달라질 수 있으므로 항상 동일한 기준이 적용되는 것은 아니라는 점을 유의해야 해서 분석하면 좋음. (e.g 주식 거래 데이터에서 volume은 음수가 허용이 안되겠지만, returns는 당연히 음수가 허용됨)

### 3-1) raw에서 사용량이 음수인 행 출력


In [5]:
np.where(raw[:, 2] < 0)

(array([3879]),)

### 3-2) 사용량이 음수인 행을 삭제하고 나머지를 raw_filter 이름으로 저장



In [6]:
raw_filter = np.delete(raw, 3879, axis=0)

### 3-3) raw_filter의 데이터 모양 확인 (삭제 진짜 되었는지)

In [7]:
raw_filter.shape

(4472, 5)

😈 사실, `raw_filter = raw[raw[:, 2] >= 0]`를 통해서 3-1)과 3-2)를 한번에 할 수도 있음

In [8]:
raw_filter2 = raw[raw[:, 2] >= 0]
raw_filter2.shape

(4472, 5)



---


## 4) 데이터를 세부적으로 확인
### 4-1) 데이터의 수집기간을 확인
동일한 날짜가 여러 행에 반복적으로 포함되어 있으므로, 데이터에 존재하는 고유한 년월(YYYYMM) 값을 추출하여 실제 데이터가 수집된 기간을 확인. 만약 데이터 수집 기간을 사전에 알고 있다면 이 과정은 생략할 수 있으며, 데이터 전처리 과정은 항상 동일한 절차로 진행되는 것이 아니라 데이터의 특성과 분석 목적에 따라 달라질 수 있음을 다시 한 번 유의해야 할 것



In [9]:
np.unique(raw_filter[:,0])

array([202001., 202002., 202003., 202004., 202005., 202006., 202007.,
       202008., 202009., 202010., 202011., 202012.])

2020년 01월 부터 2020년 12월까지가 수집기간임을 확인할 수 있음

### 4-2) 날짜별 샘플(행, row)의 갯수를 확인하기 위해 날짜와 대응되는 샘플수 출력

In [10]:
dates, counts = np.unique(raw_filter[:, 0], return_counts=True)

In [11]:
for date, count in list(zip(dates, counts)):
  print(f"{int(date)} : {int(count)}개")

202001 : 370개
202002 : 373개
202003 : 373개
202004 : 373개
202005 : 373개
202006 : 373개
202007 : 373개
202008 : 373개
202009 : 373개
202010 : 373개
202011 : 372개
202012 : 373개


월별 샘플 수가 대부분 `373개`로 거의 일정하고(202001=370, 202011=372 정도의 작은 차이만 존재) 전체 데이터 구조도 동일한 5개 변수로 반복되기 때문에 **매달 동일한 산업/시설 집단을 반복적으로 조사한 패널에 가까운 구조**라고 해석하는 것이 자연스러움. 따라서 산업 코드 등으로 개별 객체가 명시적으로 구분되지는 않았지만, 월별 샘플 수가 거의 동일하게 유지되는 점을 고려할 때 조사대상 집단은 대부분 동일한 구성으로 유지된 것으로 추정됨. 어디까지나 예상임.




---

## 5) 전체 기간의 사용량 최소값, 평균, 최대값 출력

In [12]:
usage = raw_filter[:, 2]
usage.min(), usage.mean(), usage.max() # np.min(usage), np.mean(usage), np.max(usage)도 가능함

(np.float64(0.0), np.float64(185799627.86762077), np.float64(9193268214.0))



---


## 6) 날짜에 대응되는 "가구수" 평균 출력



In [13]:
for date in dates:
  house_date = raw_filter[raw_filter[:, 0] == date, 1]
  print(f"년월: {date}, 가구수평균: {house_date.mean()}")


년월: 202001.0, 가구수평균: 29537.6
년월: 202002.0, 가구수평균: 29322.680965147454
년월: 202003.0, 가구수평균: 32294.300268096515
년월: 202004.0, 가구수평균: 29830.922252010725
년월: 202005.0, 가구수평균: 29485.195710455762
년월: 202006.0, 가구수평균: 29572.911528150133
년월: 202007.0, 가구수평균: 29646.686327077747
년월: 202008.0, 가구수평균: 29719.195710455762
년월: 202009.0, 가구수평균: 29787.973190348526
년월: 202010.0, 가구수평균: 29816.627345844503
년월: 202011.0, 가구수평균: 33011.24462365591
년월: 202012.0, 가구수평균: 30419.753351206433




---


## 7) 날짜에  대응되는 "사용량" 최소값, 평균, 최대값 출력
>*출력 예시  </br>
'년월: 202001 , 사용량최소값: 0 , 사용량평균: 100 , 사용량최대값: 100000' </br>'년월: 202002 , 사용량최소값: 0 , 사용량평균: 100 , 사용량최대값: 100000'

In [14]:
for date in dates:
  usage_date = raw_filter[raw_filter[:, 0] == date, 2]
  print(f"년월: {date}, 사용량최솟값: {usage_date.min()}, 사용량평균: {usage_date.mean()}, 사용량최대값: {usage_date.max()}")

년월: 202001.0, 사용량최솟값: 511.0, 사용량평균: 202694711.9783784, 사용량최대값: 8995713920.0
년월: 202002.0, 사용량최솟값: 534.0, 사용량평균: 191548364.22520107, 사용량최대값: 8599317103.0
년월: 202003.0, 사용량최솟값: 517.0, 사용량평균: 186852684.48793566, 사용량최대값: 8331899643.0
년월: 202004.0, 사용량최솟값: 520.0, 사용량평균: 175269887.47453085, 사용량최대값: 7778915591.0
년월: 202005.0, 사용량최솟값: 506.0, 사용량평균: 167697877.4530831, 사용량최대값: 7476966951.0
년월: 202006.0, 사용량최솟값: 494.0, 사용량평균: 175680257.4691689, 사용량최대값: 8024366854.0
년월: 202007.0, 사용량최솟값: 511.0, 사용량평균: 187335646.94369972, 사용량최대값: 8523254374.0
년월: 202008.0, 사용량최솟값: 0.0, 사용량평균: 194258433.90884718, 사용량최대값: 8794009284.0
년월: 202009.0, 사용량최솟값: 0.0, 사용량평균: 194542571.077748, 사용량최대값: 8625872422.0
년월: 202010.0, 사용량최솟값: 0.0, 사용량평균: 172735303.61930296, 사용량최대값: 7706305969.0
년월: 202011.0, 사용량최솟값: 418.0, 사용량평균: 182606879.17204303, 사용량최대값: 8289697434.0
년월: 202012.0, 사용량최솟값: 0.0, 사용량평균: 198500242.34852546, 사용량최대값: 9193268214.0




---


## 8) 날짜에 대응되는 "전기요금" 최소값, 평균, 중앙값, 최대값, 분산 출력
>*출력에시 </br>
'년월: 202001.0 , 전기요금최소값: 10 , 전기요금평균: 400 , 전기요금중앙값: 300 , 전기요금최대값: 10000 , 전기요금분산: 100' </br>
          '년월: 202002.0 , 전기요금최소값: 10 , 전기요금평균: 400 , 전기요금중앙값: 300 , 전기요금최대값: 10000 , 전기요금분산: 100'

In [15]:
for date in dates:
  bill_date = raw_filter[raw_filter[:, 0] == date, 3]
  print(f"년월: {date}, 전기요금최솟값: {bill_date.min()}, 전기요금평균: {bill_date.mean()}, 전기요금중앙값: {np.median(bill_date)}, 전기요금최대값: {bill_date.max()}, 전기요금분산: {float(bill_date.var() /1e21) }")

년월: 202001.0, 전기요금최솟값: 77965.0, 전기요금평균: 23352285788.783783, 전기요금중앙값: 3268792013.5, 전기요금최대값: 1061535709786.0, 전기요금분산: 7.048727551271482
년월: 202002.0, 전기요금최솟값: 80088.0, 전기요금평균: 22532407022.10724, 전기요금중앙값: 3119643130.0, 전기요금최대값: 1033127200639.0, 전기요금분산: 6.629747331608501
년월: 202003.0, 전기요금최솟값: 70795.0, 전기요금평균: 18887821233.683647, 전기요금중앙값: 2609605881.0, 전기요금최대값: 869048300673.0, 전기요금분산: 4.710385841341425
년월: 202004.0, 전기요금최솟값: 64704.0, 전기요금평균: 17060816646.943699, 전기요금중앙값: 2169863390.0, 전기요금최대값: 780211035574.0, 전기요금분산: 3.875001905809325
년월: 202005.0, 전기요금최솟값: 63791.0, 전기요금평균: 16482518048.648794, 전기요금중앙값: 2004116784.0, 전기요금최대값: 755695365936.0, 전기요금분산: 3.629244563973338
년월: 202006.0, 전기요금최솟값: 0.0, 전기요금평균: 20999345666.99732, 전기요금중앙값: 2576707016.0, 전기요금최대값: 977348887916.0, 전기요금분산: 5.991444122091678
년월: 202007.0, 전기요금최솟값: 84812.0, 전기요금평균: 23359593437.3941, 전기요금중앙값: 3007658240.0, 전기요금최대값: 1085637813332.0, 전기요금분산: 7.296662398129983
년월: 202008.0, 전기요금최솟값: 0.0, 전기요금평균: 23575742890.402145, 전기요금중앙값: 30



---


## 9) *6, 7, 8* 의 결과를 함수를 통해서 한번에 실행할 수도 있음
입력은 raw_filter, 출력은 년월에 따른 가구수 평균, 평균 사용량, 평균 전기요금이 되도록 함수를 정의해놓는다면, **나중에 데이터가 바뀌더라도 함수 실행만으로 바로 주요 통계값 확인 가능**


In [16]:
def show_date_means(raw_filter):
    dates = np.unique(raw_filter[:, 0])
    for date in dates:
      data_date = raw_filter[raw_filter[:, 0] == date, 1:4]
      means = data_date.mean(axis=0)
      print(f"년월: {date}, 가구수평균: {means[0]}, 사용량평균: {means[1]}, 전기요금평균: {means[2]}")

In [17]:
show_date_means(raw_filter)

년월: 202001.0, 가구수평균: 29537.6, 사용량평균: 202694711.9783784, 전기요금평균: 23352285788.783783
년월: 202002.0, 가구수평균: 29322.680965147454, 사용량평균: 191548364.22520107, 전기요금평균: 22532407022.10724
년월: 202003.0, 가구수평균: 32294.300268096515, 사용량평균: 186852684.48793566, 전기요금평균: 18887821233.683647
년월: 202004.0, 가구수평균: 29830.922252010725, 사용량평균: 175269887.47453085, 전기요금평균: 17060816646.943699
년월: 202005.0, 가구수평균: 29485.195710455762, 사용량평균: 167697877.4530831, 전기요금평균: 16482518048.648794
년월: 202006.0, 가구수평균: 29572.911528150133, 사용량평균: 175680257.4691689, 전기요금평균: 20999345666.99732
년월: 202007.0, 가구수평균: 29646.686327077747, 사용량평균: 187335646.94369972, 전기요금평균: 23359593437.3941
년월: 202008.0, 가구수평균: 29719.195710455762, 사용량평균: 194258433.90884718, 전기요금평균: 23575742890.402145
년월: 202009.0, 가구수평균: 29787.973190348526, 사용량평균: 194542571.077748, 전기요금평균: 19905331803.517426
년월: 202010.0, 가구수평균: 29816.627345844503, 사용량평균: 172735303.61930296, 전기요금평균: 16751243159.436996
년월: 202011.0, 가구수평균: 33011.24462365591, 사용량평균: 182606879.17204303, 전기요



---


## 10) VVIP 발생현황 파악
전기요금이 1조(1,000,000,000,000) 이상인 산업군를 VVIP로 정의(철강, 반도체 공장과 같은 초대형 전력 소비 산업)하고, VVIP가 나오지 않는 달은 가격 인상 후보에서 제외함. 아래 결과를 보면, 인상 후보는 1월, 2월, 7월, 8월, 12월로 주로 **여름**과 **겨울**에 집중된다는 것을 알 수 있음



In [18]:
for date in dates:
  vvip_count = (raw_filter[raw_filter[:, 0] == date, 3] > 1000000000000).sum()
  print(f"년월: {date}, vvip 수 : {vvip_count}")


년월: 202001.0, vvip 수 : 1
년월: 202002.0, vvip 수 : 1
년월: 202003.0, vvip 수 : 0
년월: 202004.0, vvip 수 : 0
년월: 202005.0, vvip 수 : 0
년월: 202006.0, vvip 수 : 0
년월: 202007.0, vvip 수 : 1
년월: 202008.0, vvip 수 : 1
년월: 202009.0, vvip 수 : 0
년월: 202010.0, vvip 수 : 0
년월: 202011.0, vvip 수 : 0
년월: 202012.0, vvip 수 : 1


---
## 11) 최종 날짜 선정
VVIP에 해당하는 산업들에 대해 1가구당 평균 사용량(사용량/가구수)과 1가구당 평균 전기요금(전기요금/가구수)을 계산하여 비교함. 이후 1가구당 평균 전기요금이 가장 큰 VVIP가 발생한 시점을 정책 적용 대상(target_month)으로 선정함.



In [19]:
vvips = raw_filter[raw_filter[:, 3] > 1000000000000 , :] ## vvip 행들로만 이루어진 배열
vvips

array([[2.02001000e+05, 1.04708500e+06, 8.99571392e+09, 1.06153571e+12,
        1.18000000e+02],
       [2.02002000e+05, 1.04812400e+06, 8.59931710e+09, 1.03312720e+12,
        1.20000000e+02],
       [2.02007000e+05, 1.05850800e+06, 8.52325437e+09, 1.08563781e+12,
        1.27000000e+02],
       [2.02008000e+05, 1.06101400e+06, 8.79400928e+09, 1.09435307e+12,
        1.24000000e+02],
       [2.02012000e+05, 1.08360000e+06, 9.19326821e+09, 1.10327108e+12,
        1.20000000e+02]])

In [20]:
per_house_bill = vvips[:, 3] / vvips[:, 1] ## vvip 행들의 1가구당 평균 전기요금 계산i
index = per_house_bill.argmax() ## 최댓값 인덱스 반환
vvips[index, 0]

np.float64(202008.0)

분석 결과 해당 조건을 만족하는 시점은 **2020년 8월**로 확인되었으며 이를 비용 인상 대상 월로 최종 결정하였음. target month를 결정하는 과정을 다시 살펴보면, VVIP가 발생한 달들을 **전력 수요가 높은 시점의 후보군**으로 간주하고, 그중에서도 **가구당 평균 전기요금이 가장 큰 VVIP가 발생한 달**을 전체 전력 수요가 가장 집중되는 시기로 선정함.

---
## 12) 정책반영 및 정책 효과 계산
### 12-1) 정책반영 (= 데이터 수정)
target_month에 해당하는 행에 대해서만 **일괄적으로 평균판매단가(5번째 컬럼)를 5% 상승**시키고, 전기요금은 사용량 × 평균판매단가 방식으로 다시 계산하여서 target_month의 전체 산업 단가 5% 인상 시 매출 증가율을 측정함.

> 데이터를 업데이트 할 때에는 기존 데이터를 수정하지 말고, 새로운 raw_filter_new를 생성하여 원본을 보전하는 것이 좋음

In [21]:
target_month = 202008                                  # 정책 적용 대상 월 지정
raw_filter_new = raw_filter.copy()                     # 원본 보존을 위해 복사본 생성
mask1 = raw_filter_new[:, 0] == target_month  # target_month에 해당 하는 행만 true 값을 가지는 불리언 배열 생성
mask2 = raw_filter_new[:, 3] > 1000000000000 # vvip에 해당하는 행만 true 값을 가지는 불리언 배열 생성
raw_filter_new[mask1, 4] *= 1.05                        # 단가 조정 (5% 증가)
raw_filter_new[mask1, 3] = raw_filter_new[mask1, 2] * raw_filter_new[mask1, 4]  # 인상된 단가로 전기요금 재계산

### 12-1) 정책 효과 계산 (= 전기요금 중가율)
정책 적용 전(raw_filter)과 정책 적용 후(raw_filter_new)의 전기요금을 비교하여 VVIP 집단의 전기요금 증가율을 계산함

$$증가율: (A - B) / B * 100$$


In [22]:
increase_rate_all = (raw_filter_new[mask1, 3].sum() - raw_filter[mask1, 3].sum()) / raw_filter[mask1, 3].sum() * 100
print(f"정책 적용시 해당 달의 전기요금 증가율: {increase_rate_all}%")

정책 적용시 해당 달의 전기요금 증가율: 5.004016671793897%


---
## 13) 합리성 판단
앞에서는 전체 산업에 대해 **5% 일괄 인상 정책**을 적용했을 때 전기요금이 얼마나 증가하는지를 계산하였음. 그러나 **VVIP 집단보다 Non-VVIP 집단의 전기요금 증가율이 더 크게 나타난다면**, 동일한 인상 정책이 산업 간 비용 부담의 형평성을 충분히 고려하지 못했을 가능성이 있음. 정부는 전력 생산자의 매출 안정성뿐 아니라 전력 소비자의 부담 구조도 함께 고려해야 하며, 특히 전력을 상대적으로 적게 소비하는 산업이 더 높은 증가율의 비용 부담을 지게 되는 상황은 정책적으로 바람직하지 않을 수 있음. 따라서 VVIP 집단과 Non-VVIP 집단의 전기요금 증가율을 비교하여 정책의 형평성과 합리성을 평가하고, 필요할 경우 집단별 차등 인상 정책이 더 적절한 대안이 될 수 있는지를 추가적으로 검토해야함.

이를 위해서 target_month에 해당하는 산업들을 전기요금이 1조 이상인 VVIP 집단과 그 외 산업인 Non-VVIP 집단으로 구분하고 집단 각각의 전기요금 증가율을 계산하여, 동일한 요금 인상이 이루어졋을 때 두 집단이 받는 비용 증가 영향이 동일한지 또는 차이가 존재하는지 확인

> ⚠️ **참고** </br>
 위의 분석 결과 현재 데이터셋에서 VVIP는 1개고, 나머지는 Non-VVIP이지만, 추후 VVIP 선정 기준이나 데이터 셋이 변경될 수 있기 때문에 두 집단 모두 n개 이상임을 가정하고 코드 작성



In [23]:
# vvip들만 포함한 2차원 배열 생성 (인상 전, 인상 후)
vvips = raw_filter[mask1 & mask2, :]
vvips_new = raw_filter_new[mask1 & mask2, :]

# vvip들의 전기요금 증가율 계산
increase_rate_vvips = (vvips_new[:, 3].sum() - vvips[:, 3].sum()) / vvips[:, 3].sum() * 100

# non_vvip들만 포함한 2차원 배열 생성 (인상 전, 인상 후)
non_vvips = raw_filter[mask1 & ~mask2, :]
non_vvips_new = raw_filter_new[mask1 & ~mask2, :]

# non_vvip들의 전기요금 증가율 계산
increase_rate_non_vvips = (non_vvips_new[:, 3].sum() - non_vvips[:, 3].sum()) / non_vvips[:, 3].sum() * 100

print(f"vvip 전기요금 증가율: {increase_rate_vvips}%, non_vvip 전기요금 증가율: {increase_rate_non_vvips}")

vvip 전기요금 증가율: 4.626197833781387%, non_vvip 전기요금 증가율: 5.057717902771226


## 14) 결과 해석 및 정책 조정

위 분석 결과를 바탕으로 정책의 합리성을 평가하고 필요할 경우 정책 조정을 수행함. 먼저 VVIP와 Non-VVIP 집단의 전기요금 증가율을 비교하여 두 집단 간 비용 부담 구조에 차이가 존재하는지 확인하고, 특정 집단의 증가율이 과도하게 높거나 낮게 나타나는 경우 정책의 형평성이 충분히 확보되지 못한 것으로 판단할 수 있음. 이러한 경우 VVIP 기준을 기존의 전기요금 1조 원에서 상향 또는 하향 조정하거나, VVIP와 Non-VVIP 집단에 서로 다른 인상률을 적용하는 방식으로 정책을 재설계함. 또한 다양한 기준값과 인상률 조합을 반복적으로 검토해야 하므로, VVIP 기준 설정, 집단 구분, 단가 조정, 전기요금 재계산 및 증가율 산출 과정을 하나의 함수로 설계하여 정책 시나리오를 자동으로 평가할 수 있도록 구현함. 즉, 12~13과정을 반복적으로 수행할 수 있도록 함수를 설계함.

In [24]:
def optimize_policy(target_month, raw_filter, vvip_standard, vvip_rate, non_vvip_rate):
  raw_filter_new = raw_filter.copy()
  mask1 = raw_filter_new[:, 0] == target_month  # target_month에 해당 하는 행만 true 값을 가지는 불리언 배열 생성
  mask2 = raw_filter_new[:, 3] > vvip_standard # vvip에 해당하는 행만 true 값을 가지는 불리언 배열 생성
  raw_filter_new[mask1 & mask2, 4] *= (1.00 + vvip_rate / 100)                    # 단가 조정 (vvip_rate만큼 증가)
  raw_filter_new[mask1 & ~mask2, 4] *= (1.00 + non_vvip_rate / 100) # 단가 조정 (non_vvip_rate만큼 증가)
  raw_filter_new[mask1, 3] = raw_filter_new[mask1, 2] * raw_filter_new[mask1, 4]  # 인상된 단가로 전기요금 재계산

  # vvip들만 포함한 2차원 배열 생성 (인상 전, 인상 후)
  vvips = raw_filter[mask1 & mask2, :]
  vvips_new = raw_filter_new[mask1 & mask2, :]

  # vvip들의 전기요금 증가율 계산
  increase_rate_vvips = (vvips_new[:, 3].sum() - vvips[:, 3].sum()) / vvips[:, 3].sum() * 100

  # non_vvip들만 포함한 2차원 배열 생성 (인상 전, 인상 후)
  non_vvips = raw_filter[mask1 & ~mask2, :]
  non_vvips_new = raw_filter_new[mask1 & ~mask2, :]

  # non_vvip들의 전기요금 증가율 계산
  increase_rate_non_vvips = (non_vvips_new[:, 3].sum() - non_vvips[:, 3].sum()) / non_vvips[:, 3].sum() * 100

  difference = increase_rate_vvips - increase_rate_non_vvips

  if (difference > 0):
    print(f"vvip 전기요금 증가율이 non_vvip 전기요금 증가율보다 {difference:.2f}만큼 큽니다.")
  elif (difference < 0):
    print(f"non_vvip 전기요금 증가율이 vvip 전기요금 증가율보다 {-difference:.2f}만큼 큽니다.")
  else:
    print(f"두 집단의 전기요금 증가율이 {increase_rate_vvips}로 동일합니다.")

In [25]:
# 단일 요금 인상율 적용 결과
optimize_policy(202008, raw_filter, 1000000000000, 5, 5)

# 차등 요금 인상율 적용 결과
optimize_policy(202008, raw_filter, 1000000000000, 5.5, 4.5)

non_vvip 전기요금 증가율이 vvip 전기요금 증가율보다 0.43만큼 큽니다.
vvip 전기요금 증가율이 non_vvip 전기요금 증가율보다 0.57만큼 큽니다.


## 15) 결론 및 제언
본 분석은 산업별 전력 사용 데이터를 기반으로 전기요금 정책이 산업 집단에 미치는 영향을 정량적으로 평가할 수 있는 데이터 기반 정책 분석 프레임워크를 제시하였다. 먼저 전기요금 규모를 기준으로 초대형 전력 소비 산업(VVIP)을 정의하고, 월별 전력 사용 구조를 분석하여 정책 적용 시점을 선정하였다. 이후 평균판매단가 조정이라는 정책 시나리오를 적용하고 전기요금을 재계산함으로써 정책 적용 전후의 전기요금 변화와 집단별 증가율을 비교하였다. 이러한 과정을 통해 동일한 요금 인상 정책이 산업 집단 간 비용 부담 구조에 서로 다른 영향을 미칠 수 있음을 확인할 수 있었으며, 특히 전력 소비 규모가 작은 산업 집단이 상대적으로 더 높은 증가율의 부담을 지게 될 가능성도 존재함을 확인하였다.

이 분석의 의의는 단순히 특정 정책 시나리오의 결과를 계산하는 데 그치지 않고, 전력 소비 구조 분석, VVIP 기준 설정, 정책 적용, 전기요금 재계산, 집단별 증가율 비교의 전 과정을 하나의 분석 흐름으로 구조화했다는 점에 있다. 즉 본 접근 방식은 특정 수치 결과 자체보다도 다양한 정책 시나리오를 반복적으로 실험하고 평가할 수 있는 **정책 시뮬레이션 프레임워크**로서의 의미를 가진다. 정책 담당자는 VVIP 기준값, 산업 집단 구분 방식, 평균판매단가 인상률 등의 파라미터를 조정하면서 여러 정책 조합을 비교할 수 있으며, 이를 통해 전력 소비 구조와 산업 간 형평성을 동시에 고려한 합리적인 요금 정책을 설계할 수 있다.

향후 연구에서는 단순한 단가 조정 시뮬레이션을 넘어 전력 수요 변화, 산업별 전력 사용 탄력성, 계절적 전력 수요 패턴 등을 함께 고려하는 방향으로 프레임워크를 확장할 수 있다. 또한 머신러닝 기반 수요 예측 모델이나 정책 최적화 모델을 결합한다면 전력 수요 변화에 대응하는 동적인 요금 정책 설계에도 활용될 수 있을 것이다. 이러한 확장은 데이터 기반 의사결정을 강화하고 전력 산업의 안정적인 운영과 정책 효율성을 동시에 높이는 데 기여할 수 있을 것으로 기대된다.
